# MRAG Ablation Study — Data Collection (v3)

Six leave-one-out ablations against the 150-question MUTCD-150 gold set. **No GPT calls.** Per-question output in your benchmark format, everything else to `/content/ablation_results/ablation.log`.

**Fixed in v3 (vs v2):**
- `chunks_used` and `pages_used` now populated correctly (v2 used wrong dict keys — those fields were empty in all previous runs)
- Qdrant client workaround baked in (init_pipeline sometimes leaves store client blind — auto-detected and fixed)
- IPython display + stdout + stderr silenced around `ask()` (no interactive noise)
- Per-question print in your existing benchmark format: `[N/M] RUN  <ablation_id> <qa_id> r1   OK <ms> ms — <preview>`
- Pipeline is initialized ONCE and reused across ablations (avoids ~6× model reload)
- NEW: post-processing cell that backfills chunks/pages onto existing runs so retrieval metrics (Recall@5, MRR, nDCG, context_precision, retrieval_25, full 97-point) become computable

**Order of cells:** setup → keys → config → sanity → restore Qdrant → pipeline init → silence → helper → smoke test → A1–A6 → run all remaining → post-process → handoff.

**Any single ablation cell (A1–A6) is independently re-runnable and resume-safe.**


## 1. Environment setup

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
print('Colab detected:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    else:
        print('Drive already mounted')

REPO_URL = 'https://github.com/hannanazad/gemsrag.git'
REPO_DIR = '/content/gemsrag'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Repo ready at', REPO_DIR)


## 2. Install dependencies

Installs the main `mrag/` requirements plus the ablation harness extras (anthropic for Fable 5, pyyaml for config files).

In [ ]:
# Main pipeline deps (from repo requirements.txt if present)
!pip install -q -r /content/gemsrag/requirements.txt 2>/dev/null || echo 'no requirements.txt — skipping'
# Ablation harness + Fable 5 (Claude via Anthropic API)
!pip install -q anthropic pyyaml


## 3. API keys (VLM providers only — no OpenAI)

Loads from Colab Secrets. Only VLM provider keys are needed; the harness does not call GPT.

In [ ]:
try:
    from google.colab import userdata
    for k in ('DASHSCOPE_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'QWEN'):
        try:
            v = userdata.get(k)
            if v:
                target = 'DASHSCOPE_API_KEY' if k == 'QWEN' else k
                os.environ[target] = v
                print(f'  {k} -> {target}: loaded')
        except Exception:
            pass
except ImportError:
    print('Not in Colab; expecting env vars already set.')

print()
for k in ('ANTHROPIC_API_KEY', 'DASHSCOPE_API_KEY', 'GEMINI_API_KEY'):
    print(f'  {k}: {"set" if os.environ.get(k) else "NOT set"}')


## 4. Configuration

Edit these paths. Everything downstream reads from these variables.

In [ ]:
from pathlib import Path
import os

# ---------------------------------------------------------------------------
# PATHS
#
# The MUTCD-150 benchmark ships inside this repository, so the only thing you
# normally have to supply is the prebuilt Qdrant vector store.
#
# If you do not have a prebuilt store, build one first with notebook
# 01_quickstart_ask.ipynb (section 1.5) or `python scripts/ingest_v4.py`.
# See docs/REPRODUCTION_GUIDE.md for the full walkthrough.
# ---------------------------------------------------------------------------

# Repo root. On Colab the setup cell clones to /content/gemsrag; running
# locally it falls back to the current directory.
REPO_DIR = Path(os.environ.get("GEMSRAG_ROOT", "/content/gemsrag"))
if not (REPO_DIR / "benchmark").is_dir():
    REPO_DIR = Path.cwd()

# Benchmark gold file bundled with this repo (150 questions).
GOLD_QA_PATH = str(REPO_DIR / "benchmark" / "gold" / "mutcd_benchmark_gold_v1_1_msdi.jsonl")

# Prebuilt Qdrant vector store, packaged as a .tar.
# Override with the GEMSRAG_QDRANT_TAR environment variable.
QDRANT_TAR_PATH = os.environ.get(
    "GEMSRAG_QDRANT_TAR",
    "/content/drive/MyDrive/MRAG/qdrant_db.tar",
)

# Where the finished per-ablation result zips are written.
# Default is inside the repo. Point this at Google Drive if you want the
# results to survive the end of a Colab session, for example:
#   RESULTS_DRIVE_DIR = "/content/drive/MyDrive/MRAG/ablations"
RESULTS_DRIVE_DIR = os.environ.get("GEMSRAG_RESULTS", str(REPO_DIR / "runs" / "ablations"))

# ---------------------------------------------------------------------------
# RUN SETTINGS
# ---------------------------------------------------------------------------
VLM_ALIAS    = "frontier_claude"   # see CFG.list_vlm_models() for the catalog
PROMPT_STYLE = "fewshot"

# For a quick trial set this to 5. None runs all 150 questions.
LIMIT_N_QUESTIONS = None

# True = retrieval only, no answer generation. Minutes per ablation instead of
# about 70 minutes, and useful as a cheap first pass.
RETRIEVAL_ONLY = False

# --- computed ---
RESULTS_DRIVE_DIR = Path(RESULTS_DRIVE_DIR)
RESULTS_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RESULTS_DIR = Path("/content/ablation_results")
if not Path("/content").is_dir():
    LOCAL_RESULTS_DIR = REPO_DIR / "runs" / "ablation_scratch"
LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root      :", REPO_DIR)
print("Gold benchmark :", GOLD_QA_PATH)
print("Qdrant tar     :", QDRANT_TAR_PATH)
print("Local scratch  :", LOCAL_RESULTS_DIR)
print("Results out    :", RESULTS_DRIVE_DIR)
print("VLM alias      :", VLM_ALIAS)
print("Prompt style   :", PROMPT_STYLE)
print("Question limit :", LIMIT_N_QUESTIONS or "all 150")


## 5. Sanity check the gold file

In [ ]:
import json

def _count(path):
    return sum(1 for l in open(path) if l.strip())
def _first_keys(path):
    with open(path) as f:
        for l in f:
            if l.strip():
                return list(json.loads(l).keys())
    return []

assert Path(GOLD_QA_PATH).exists(), f'gold file not found: {GOLD_QA_PATH}'
print('gold_qa.jsonl rows :', _count(GOLD_QA_PATH))
print('first-record keys  :', _first_keys(GOLD_QA_PATH))
assert Path(QDRANT_TAR_PATH).exists(), f'qdrant tar not found: {QDRANT_TAR_PATH}'
print('qdrant tar size    :', Path(QDRANT_TAR_PATH).stat().st_size, 'bytes')


## 6. Restore Qdrant store from tar (idempotent)

Colab local disk is not persistent. Extract the tar if `/content/qdrant_db/collection/` is empty; skip otherwise.

In [ ]:
import os, subprocess
from mrag.config import CFG

qdrant_dir = str(CFG.qdrant_dir)
coll_dir   = os.path.join(qdrant_dir, 'collection')
has_coll   = os.path.isdir(coll_dir) and bool(os.listdir(coll_dir))

if not has_coll:
    print(f'Qdrant store empty at {qdrant_dir} — extracting from tar (~2-5 min)')
    subprocess.run(['rm', '-rf', qdrant_dir], check=False)
    subprocess.run(['tar', '-xf', QDRANT_TAR_PATH, '-C', '/content/'], check=True)
    print('collections:', os.listdir(coll_dir))
else:
    print(f'Qdrant already populated at {qdrant_dir}, {len(os.listdir(coll_dir))} collections')


## 7. Pipeline init (once per session)

Initializes VLM, embedders, KG, reranker, retriever. Applies the Qdrant client workaround baked into `run_ablation.py` on first sweep call, but we also verify here.

In [ ]:
import gc
try:
    del pipeline
except NameError:
    pass
gc.collect()

from mrag.config import CFG
CFG.set_vlm_model(VLM_ALIAS)
CFG.set_answer_style(PROMPT_STYLE)
print('VLM       :', getattr(CFG, 'vlm_model_api', '?'))
print('Answer    :', getattr(CFG, 'prompt_style_answer', '?'))

try:
    from mrag.pipeline import init_pipeline
except ImportError:
    from mrag.ask import init_pipeline
pipeline = init_pipeline()

# Verify store client sees the collections. If not, replace with a fresh QdrantClient.
if hasattr(pipeline, 'store') and hasattr(pipeline.store, '_client'):
    try:
        colls = pipeline.store._client.get_collections().collections
    except Exception:
        colls = []
    if not colls:
        from qdrant_client import QdrantClient
        try: pipeline.store._client.close()
        except Exception: pass
        pipeline.store._client = QdrantClient(path=str(CFG.qdrant_dir))
        colls = pipeline.store._client.get_collections().collections
        print('replaced stale store client with fresh QdrantClient')
    print('collections visible:', [c.name for c in colls])

print('pipeline    :', type(pipeline).__name__)
print('rerank type :', type(getattr(pipeline, 'rerank', None)).__name__)


## 8. Silence library noise + mrag.ask() display output

Route Python logs to file; silence transformers/PIL/torch warnings; capture mrag.ask()'s IPython display output so per-question progress lines stay clean.

In [ ]:
import warnings, logging
from pathlib import Path

warnings.filterwarnings('ignore')
for name in ('transformers', 'PIL', 'torch', 'urllib3', 'httpx', 'mrag'):
    logging.getLogger(name).setLevel(logging.ERROR)

LOG_FILE = Path('/content/ablation_results/ablation.log')
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

ab_log = logging.getLogger('ablations')
ab_log.setLevel(logging.INFO)
ab_log.propagate = False
for h in list(ab_log.handlers):
    ab_log.removeHandler(h)
fh = logging.FileHandler(str(LOG_FILE), mode='a')
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s'))
ab_log.addHandler(fh)
sh = logging.StreamHandler()
sh.setLevel(logging.ERROR)
sh.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
ab_log.addHandler(sh)

# Silence mrag.ask.ask() — captures stdout, stderr, and IPython display.
import io, contextlib, mrag.ask
from IPython.utils.io import capture_output
if not hasattr(mrag.ask, '_original_ask_pre_quiet'):
    mrag.ask._original_ask_pre_quiet = mrag.ask.ask
_real_ask = mrag.ask._original_ask_pre_quiet
def _silent_ask(question):
    with capture_output(stdout=True, stderr=True, display=True):
        return _real_ask(question)
mrag.ask.ask = _silent_ask

print(f'Silent mode ON. Full logs at: {LOG_FILE}')


## 9. `run_and_package` helper

One call per ablation. Runs the sweep, packages results into a zip on Drive. Resume-safe.

In [ ]:
from ablations.ablation_shims import ABLATIONS, list_ablations
from ablations.run_ablation import run_ablation_sweep
from ablations.package_results import package_ablation

print('Registered ablations:')
for a in list_ablations():
    print(f'  {a["id"]:20s} — {a["description"]}')

def run_and_package(ablation_id, retrieval_only=None, limit=None):
    retrieval_only = RETRIEVAL_ONLY if retrieval_only is None else retrieval_only
    limit = LIMIT_N_QUESTIONS if limit is None else limit

    runs_path = LOCAL_RESULTS_DIR / f'runs_{ablation_id}.jsonl'
    zip_path  = RESULTS_DRIVE_DIR / f'mrag_ablation_{ablation_id}.zip'

    counters = run_ablation_sweep(
        ablation_id=ablation_id, gold_path=Path(GOLD_QA_PATH),
        output_path=runs_path, vlm_alias=VLM_ALIAS,
        prompt_style=PROMPT_STYLE, limit=limit,
        retrieval_only=retrieval_only, pipeline=pipeline,
    )
    ablation_meta = ABLATIONS.get(ablation_id)
    package_ablation(
        ablation_id=ablation_id, runs_path=runs_path,
        zip_output_path=zip_path, vlm_alias=VLM_ALIAS,
        prompt_style=PROMPT_STYLE, repo_dir=Path(REPO_DIR),
        description=ablation_meta.description if ablation_meta else '',
    )
    print(f'→ {ablation_id}: total={counters["total"]} ok={counters["ok"]} err={counters["err"]} → {zip_path.name}')
    return zip_path


## 10. Smoke test (optional — 5 Qs, ~2 min)

In [ ]:
smoke_runs = LOCAL_RESULTS_DIR / 'SMOKE_runs_A1_no_router.jsonl'
if smoke_runs.exists():
    smoke_runs.unlink()
counters = run_ablation_sweep(
    ablation_id='A1_no_router', gold_path=Path(GOLD_QA_PATH),
    output_path=smoke_runs, vlm_alias=VLM_ALIAS,
    prompt_style=PROMPT_STYLE, limit=5, retrieval_only=RETRIEVAL_ONLY,
    pipeline=pipeline,
)
print('Smoke:', counters)


## 11. Ablation A1_no_router

`CFG.use_question_router = False`. Runs 150 Qs → zips to `mrag_ablation_A1_no_router.zip` on Drive. Re-run only this cell if it fails or partially completes — resume-safe.

In [ ]:
zip_A1 = run_and_package('A1_no_router')


## 12. Ablation A2_no_vlm_filter

`CFG.use_vlm_figure_filter = False`. Runs 150 Qs → zips to `mrag_ablation_A2_no_vlm_filter.zip` on Drive. Re-run only this cell if it fails or partially completes — resume-safe.

In [ ]:
zip_A2 = run_and_package('A2_no_vlm_filter')


## 13. Ablation A3_no_graph

`CFG.w_graph = 0.0`. Runs 150 Qs → zips to `mrag_ablation_A3_no_graph.zip` on Drive. Re-run only this cell if it fails or partially completes — resume-safe.

In [ ]:
zip_A3 = run_and_package('A3_no_graph')


## 14. Ablation A4_no_rule_type

`CFG.w_ruletype = 0.0`. Runs 150 Qs → zips to `mrag_ablation_A4_no_rule_type.zip` on Drive. Re-run only this cell if it fails or partially completes — resume-safe.

In [ ]:
zip_A4 = run_and_package('A4_no_rule_type')


## 15. Ablation A5_no_hierarchy

`CFG.w_hierarchy = 0.0`. Runs 150 Qs → zips to `mrag_ablation_A5_no_hierarchy.zip` on Drive. Re-run only this cell if it fails or partially completes — resume-safe.

In [ ]:
zip_A5 = run_and_package('A5_no_hierarchy')


## 16. Ablation A6_no_reranker

`pipeline.rerank` and `pipeline.retriever.rerank` → `NoOpReranker` at runtime. Runs 150 Qs → zips to `mrag_ablation_A6_no_reranker.zip` on Drive. Re-run only this cell if it fails or partially completes — resume-safe.

In [ ]:
zip_A6 = run_and_package('A6_no_reranker')


## 17. Run all remaining (convenience)

Runs any of A1–A6 that don't already have a zip on Drive. Skip if you're running cells individually.

In [ ]:
ALL_ABLATIONS = ['A1_no_router', 'A2_no_vlm_filter', 'A3_no_graph',
                 'A4_no_rule_type', 'A5_no_hierarchy', 'A6_no_reranker']
for aid in ALL_ABLATIONS:
    zpath = RESULTS_DRIVE_DIR / f'mrag_ablation_{aid}.zip'
    if zpath.exists():
        print(f'skip {aid} — {zpath.name} already exists')
        continue
    run_and_package(aid)


## 18. Post-process: backfill retrieval metrics

v2 of the harness did not correctly capture `chunks_used` and `pages_used` — the wrong dict keys were used against mrag's `ask()` return. This cell fixes the completed runs by re-calling `pipeline.retriever.retrieve()` (retrieval only — no generation, ~200 ms per question) with each ablation applied, and merges the chunks/pages back into the existing records.

**Runs on already-completed jsonl files** — total ~3 min for six ablations.

After this cell, re-run cell 17 or re-package the zips manually so the fixed runs make it back to Drive.


In [ ]:
from ablations.postprocess_retrieval import postprocess_all

results = postprocess_all(
    ablations_dir=LOCAL_RESULTS_DIR,
    pipeline=pipeline,
    force=False,   # set True to overwrite even records that already have chunks/pages
)

print('\nPost-processing summary:')
for aid, c in results.items():
    print(f'  {aid:20s}  filled={c["filled"]:3d}  skipped={c["skipped"]:3d}  err={c["err"]:3d}  total={c["total"]:3d}')


### 18b. Re-package the zips with backfilled retrieval

In [ ]:
# Re-zip so the updated runs.jsonl (with chunks/pages) lands on Drive.
from ablations.ablation_shims import ABLATIONS
for aid in results.keys():
    runs_path = LOCAL_RESULTS_DIR / f'runs_{aid}.jsonl'
    zip_path  = RESULTS_DRIVE_DIR / f'mrag_ablation_{aid}.zip'
    ablation_meta = ABLATIONS.get(aid)
    package_ablation(
        ablation_id=aid, runs_path=runs_path, zip_output_path=zip_path,
        vlm_alias=VLM_ALIAS, prompt_style=PROMPT_STYLE,
        repo_dir=Path(REPO_DIR),
        description=ablation_meta.description if ablation_meta else '',
    )
    print(f'  re-packaged {aid} → {zip_path.name}')


## 19. Handoff to GPT

The zips on Drive at `RESULTS_DRIVE_DIR` now contain complete records:

- `runs_<id>.jsonl` — qa_id, question, RAG answer, chunks_used (backfilled), figures_used, pages_used (backfilled), debug, latency
- `meta.json` — ablation config, timestamp, git sha, counters
- `README.txt` — brief handoff note

Hand each zip to GPT-5.6 with your MUTCD-150 rubric prompt. All retrieval metrics (Recall@5, MRR, nDCG@6, context_precision@6, retrieval_25) should now be computable.

This harness produced no scores, no aggregations, no interpretation.
